In [1]:
import os 
import numpy as np 
import pandas as pd 
from absl import app 
from absl import flags 
import compress_pickle
from typing import Sequence 
import matplotlib.pyplot as plt

In [2]:
params_list = [] 
all_params_list = [] 
ks = [x for x in range(1,11)] 
# ks = [x for x in range(120,501, 20)]
# ks.extend([x for x in range(20,1001, 20)])
ks.extend([x for x in range(20,501, 20)])
# ks.extend([x for x in range(20,101, 20)])

def get_n_k_for_num_ratings(all_values, num_ratings=5000): 
    values=[] 
    for x in all_values:
        if x==0:
            x=1 
        n = int(np.floor(num_ratings/x)) 
        if n > 0:
            values.append((n, int(x))) 
    return values


# vals = get_n_k_for_num_ratings(ks) 
# [x[1] for x in vals], [x[0] for x in vals]
ratings_list = []
# nk_list = [2500]
# nk_list = [5000]
# nk_list = [2500, 5000, 10000, 25000, 50000]
# nk_list = [1000, 2500, 5000, 10000, 25000, 50000]
# nk_list = [100, 250, 500, 1000, 2500]
nk_list = [100, 250, 500, 1000, 2500, 5000, 10000, 25000, 50000]
# for r in range(1000, 5001, 1000):
# for r in range(5000, 5001, 500): 
for r in nk_list:
    params = get_n_k_for_num_ratings(ks, r)
    params_list.append(params)
    all_params_list.extend(params)
    ratings_list.extend([(r-x[0]*x[1]) for x in params])
    print(params)

[(100, 1), (50, 2), (33, 3), (25, 4), (20, 5), (16, 6), (14, 7), (12, 8), (11, 9), (10, 10), (5, 20), (2, 40), (1, 60), (1, 80), (1, 100)]
[(250, 1), (125, 2), (83, 3), (62, 4), (50, 5), (41, 6), (35, 7), (31, 8), (27, 9), (25, 10), (12, 20), (6, 40), (4, 60), (3, 80), (2, 100), (2, 120), (1, 140), (1, 160), (1, 180), (1, 200), (1, 220), (1, 240)]
[(500, 1), (250, 2), (166, 3), (125, 4), (100, 5), (83, 6), (71, 7), (62, 8), (55, 9), (50, 10), (25, 20), (12, 40), (8, 60), (6, 80), (5, 100), (4, 120), (3, 140), (3, 160), (2, 180), (2, 200), (2, 220), (2, 240), (1, 260), (1, 280), (1, 300), (1, 320), (1, 340), (1, 360), (1, 380), (1, 400), (1, 420), (1, 440), (1, 460), (1, 480), (1, 500)]
[(1000, 1), (500, 2), (333, 3), (250, 4), (200, 5), (166, 6), (142, 7), (125, 8), (111, 9), (100, 10), (50, 20), (25, 40), (16, 60), (12, 80), (10, 100), (8, 120), (7, 140), (6, 160), (5, 180), (5, 200), (4, 220), (4, 240), (3, 260), (3, 280), (3, 300), (3, 320), (2, 340), (2, 360), (2, 380), (2, 400), (

In [3]:
len(params_list), len(ratings_list), len(all_params_list)

(9, 282, 282)

In [4]:
def gather_data(_N_ITEMS, _K_RESPONSES, distortion_values, exp_dir, metrics_list, _M_CATEGORIES, actual_p=False): 
    final_table = pd.DataFrame() 
    for distortion in distortion_values:
        file_path = f'{exp_dir}results_N={_N_ITEMS}_K={_K_RESPONSES}_cat_responses_simulated_distr_dist={distortion}_gen_N={_N_ITEMS}_K={_K_RESPONSES}_M={_M_CATEGORIES}_num_samples=1000.pkl.csv'
        if actual_p:
            file_path = f'{exp_dir}results_N={_N_ITEMS}_K={_K_RESPONSES}_cat_actual_responses_simulated_distr_dist={distortion}_gen_N={_N_ITEMS}_K={_K_RESPONSES}_M={_M_CATEGORIES}_num_samples=1000.pkl.csv'
        experiment_results = pd.read_csv(file_path)
        
        intermediate_table = pd.DataFrame() 
        intermediate_table['$\\Delta$'] = (experiment_results['M2 GT Alt'] - experiment_results['M1 GT Alt']).abs() 
        intermediate_table['p-value'] = experiment_results['GT_Pvalue'] 
        intermediate_table['M1 GT Alt'] = experiment_results['M1 GT Alt']
        intermediate_table['M2 GT Alt'] = experiment_results['M2 GT Alt']
        # intermediate_table['Metric'] = ['$\\Gamma_{\\rm Accuracy}$', '$\\Gamma_{\\rm F1-score}$']  
        intermediate_table['Metric'] = metrics_list 
        # intermediate_table['Metric'] = ['Accuracy'] 
        intermediate_table[f'$\\epsilon$'] = distortion 
        final_table = pd.concat([final_table, intermediate_table]) 
 
    final_table = final_table.melt(["Metric", "$\\epsilon$"]).sort_values(by=["Metric","variable"]).pivot(index = "$\\epsilon$", columns=["Metric","variable"]) 
    # final_table = final_table.reset_index(drop=True) 
    final_table = final_table.reset_index() 
    final_table.columns = pd.MultiIndex.from_tuples([(j,k) for i,j,k in final_table.columns]) 
    final_table.columns = ['\\_'.join(col) for col in final_table.columns] 
    final_table["N"] = pd.Series([_N_ITEMS]*len(distortion_values)) 
    final_table["K"] = pd.Series([_K_RESPONSES]*len(distortion_values)) 
    # final_table["NxK"] = final_table["N"]*final_table["K"] 
 
    return final_table

In [5]:
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_dices/", "dataset": "DICES", "num_categories": "3",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_d3code/", "dataset": "D3code", "num_categories": "2",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_jobsQ1/", "dataset": "JobsQ1", "num_categories": "5",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_jobsQ3/", "dataset": "JobsQ3", "num_categories": "12",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_toxicity/", "dataset": "Toxicity", "num_categories": "2",}

# dataset_info = {"exp_dir": "../ptest_arr_uniform/", "dataset": "uniform", "num_categories": "2",}
# dataset_info = {"exp_dir": "../ptest_arr_gamma/", "dataset": "gamma", "num_categories": "3",}

datasets = [{"exp_dir": "../../../../data/ptest_arr_toxicity/", "dataset": "Toxicity", "num_categories": "2",},
            {"exp_dir": "../../../../data/ptest_arr_dices/", "dataset": "DICES", "num_categories": "3",},
            {"exp_dir": "../../../../data/ptest_arr_d3code/", "dataset": "D3code", "num_categories": "2",},
            {"exp_dir": "../../../../data/ptest_arr_jobsQ1/", "dataset": "JobsQ1", "num_categories": "5",},
            {"exp_dir": "../../../../data/ptest_arr_jobsQ3/", "dataset": "JobsQ3", "num_categories": "12",},]

# datasets = [{"exp_dir": "../ptest_arr_uniform/", "dataset": "Balanced (M=2)", "num_categories": "2",},
#             {"exp_dir": "../ptest_arr_uniform/", "dataset": "Balanced (M=3)", "num_categories": "3",},
#             {"exp_dir": "../ptest_arr_uniform/", "dataset": "Balanced (M=4)", "num_categories": "4",},
#             {"exp_dir": "../ptest_arr_uniform/", "dataset": "Balanced (M=5)", "num_categories": "5",},
#             {"exp_dir": "../../../../shared/rc/populaltetion/ptest_arr_uniform/", "dataset": "Balanced (M=12)", "num_categories": "12",},
#             {"exp_dir": "../ptest_arr_gamma/", "dataset": "Unbalanced (M=2)", "num_categories": "2",},
#             {"exp_dir": "../ptest_arr_gamma/", "dataset": "Unbalanced (M=3)", "num_categories": "3",},
#             {"exp_dir": "../ptest_arr_gamma/", "dataset": "Unbalanced (M=4)", "num_categories": "4",},
#             {"exp_dir": "../ptest_arr_gamma/", "dataset": "Unbalanced (M=5)", "num_categories": "5",},
#             {"exp_dir": "../../../../shared/rc/population/ptest_arr_gamma/", "dataset": "Unbalanced (M=12)", "num_categories": "12",},]

In [6]:
col = "K"
distortion = 0.3
metrics_list = ['Accuracy', 'MAE', 'Wins', 'KL-Div']

confidence_level = 0.95
nk_list_idx=0

In [7]:
# all_dfs = {}
all_dfs = []

for dataset_info in datasets:
    _M_CATEGORIES = dataset_info['num_categories']
    exp_dir = dataset_info['exp_dir']
    dataset = dataset_info['dataset']

    errors = []
    dfs_nk_list = []
    for i, nks in enumerate(params_list):
        data_df_list = []
        # for n,k in params_list[nk_list_idx][:35]:
        for n,k in nks[:35]:
            try:
                data_df = gather_data(n, k, [distortion], exp_dir, metrics_list, _M_CATEGORIES)
                data_df_list.append(data_df)
            except FileNotFoundError:
                errors.append((n,k))
                # print(f"File not found!: {n,k}")
            except:
                errors.append((n,k))
                print(f"Some exception occured!: {n,k}")

        if data_df_list:
            df_ratings_nk = pd.concat(data_df_list)
            df_ratings_nk = df_ratings_nk.reset_index(drop=True)
            df_ratings_nk["NK"] = nk_list[i]
            dfs_nk_list.append(df_ratings_nk)

    # # all_dfs[dataset] = dfs_nk_list
    # all_dfs[dataset] = pd.concat(dfs_nk_list, ignore_index=True)

    all_dfs.append(pd.concat(dfs_nk_list, ignore_index=True))

    print(f"Distortion: {distortion}, Error list len: {len(errors)}")

    # break

print(len(all_dfs))

Distortion: 0.3, Error list len: 0
Distortion: 0.3, Error list len: 0
Distortion: 0.3, Error list len: 0
Distortion: 0.3, Error list len: 0
Distortion: 0.3, Error list len: 0
5


In [8]:
all_dfs[0]

,\_,Accuracy\_$\Delta$,Accuracy\_M1 GT Alt,Accuracy\_M2 GT Alt,Accuracy\_p-value,KL-Div\_$\Delta$,KL-Div\_M1 GT Alt,KL-Div\_M2 GT Alt,KL-Div\_p-value,MAE\_$\Delta$,MAE\_M1 GT Alt,MAE\_M2 GT Alt,MAE\_p-value,Wins\_$\Delta$,Wins\_M1 GT Alt,Wins\_M2 GT Alt,Wins\_p-value,N,K,NK
0,0.3,0.038909,0.633574,0.594665,0.351775,1.075093,10.124736,11.199829,0.346428,0.038909,0.366426,0.405335,0.351152,3.890891,22.395395,18.504505,0.357980,100,1,100
1,0.3,0.035275,0.712833,0.677558,0.395049,0.541277,5.163071,5.704348,0.410898,0.036727,0.290050,0.326777,0.338837,2.876877,14.328328,11.451451,0.370760,50,2,100
2,0.3,0.051112,0.710043,0.658932,0.376271,0.115171,3.157485,3.272656,0.470547,0.033963,0.247844,0.281807,0.328930,2.313313,10.762763,8.449449,0.370656,33,3,100
3,0.3,0.046647,0.741702,0.695055,0.422238,0.012256,2.080441,2.068185,0.487681,0.034094,0.219560,0.253654,0.341411,2.023023,8.779780,6.756757,0.389501,25,4,100
4,0.3,0.062963,0.752252,0.689289,0.385559,0.121193,1.499795,1.378602,0.466709,0.033604,0.198348,0.231952,0.327603,1.863864,7.472472,5.608609,0.383722,20,5,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277,0.3,0.129760,0.967514,0.837754,0.000272,0.043262,0.002483,0.045745,0.000000,0.088004,0.023019,0.111023,0.000000,89.226226,103.315315,14.089089,0.000000,119,420,50000
278,0.3,0.132345,0.968783,0.836438,0.000554,0.043249,0.002374,0.045623,0.000000,0.088531,0.022389,0.110920,0.000000,85.522523,98.561562,13.039039,0.000000,113,440,50000
279,0.3,0.129352,0.968747,0.839395,0.001217,0.043558,0.002258,0.045816,0.000000,0.089263,0.021874,0.111137,0.000000,82.511512,94.610611,12.099099,0.000000,108,460,50000
280,0.3,0.134134,0.969700,0.835566,0.000451,0.043293,0.002151,0.045444,0.000000,0.089416,0.021514,0.110930,0.000000,79.858859,91.292292,11.433433,0.000000,104,480,50000


In [33]:
metric = 'MAE'
ls = all_dfs[-1][f'{metric}\\_p-value']
met_df = all_dfs[-1].iloc[ls.index[ls<=0.05]]
min_nk = met_df['NK'].min()
nk_df = met_df[met_df['NK']==min_nk]
# min_k = nk_df['K'].min()
nk_df


,\_,Accuracy\_$\Delta$,Accuracy\_M1 GT Alt,Accuracy\_M2 GT Alt,Accuracy\_p-value,KL-Div\_$\Delta$,KL-Div\_M1 GT Alt,KL-Div\_M2 GT Alt,KL-Div\_p-value,MAE\_$\Delta$,MAE\_M1 GT Alt,MAE\_M2 GT Alt,MAE\_p-value,Wins\_$\Delta$,Wins\_M1 GT Alt,Wins\_M2 GT Alt,Wins\_p-value,N,K,NK
26,0.3,0.392559,0.734568,0.342009,0.157857,0.539278,1.507314,2.046592,0.314392,0.015768,0.041911,0.057679,0.040713,3.956957,4.797798,0.840841,0.121712,6,40,250
27,0.3,0.511762,0.847097,0.335335,0.151103,0.363456,0.895016,1.258472,0.329685,0.017888,0.034800,0.052688,0.029206,3.134134,3.494494,0.360360,0.135621,4,60,250
28,0.3,0.535536,0.908575,0.373040,0.164762,0.280868,0.613888,0.894757,0.338540,0.018972,0.030113,0.049085,0.027255,2.551552,2.746747,0.195195,0.170100,3,80,250
29,0.3,0.573574,0.945445,0.371872,0.253266,0.252019,0.478128,0.730147,0.328402,0.020282,0.027046,0.047328,0.034455,1.797798,1.886887,0.089089,0.263171,2,100,250
30,0.3,0.614114,0.969469,0.355355,0.215032,0.197261,0.371565,0.568826,0.343073,0.021242,0.024774,0.046016,0.021148,1.881882,1.934935,0.053053,0.252328,2,120,250
31,0.3,0.623624,0.983984,0.360360,0.393979,0.229352,0.289187,0.518539,0.334898,0.022586,0.022631,0.045217,0.046757,0.964965,0.978979,0.014014,0.486131,1,140,250
32,0.3,0.622623,0.991992,0.369369,0.393768,0.220031,0.240962,0.460993,0.312309,0.021722,0.022114,0.043836,0.045170,0.964965,0.979980,0.015015,0.473767,1,160,250
33,0.3,0.664665,0.992993,0.328328,0.353706,0.225563,0.185788,0.411351,0.268857,0.023932,0.019869,0.043801,0.027798,0.987988,0.992993,0.005005,0.491139,1,180,250
34,0.3,0.639640,0.994995,0.355355,0.382896,0.218724,0.161637,0.380361,0.243801,0.023856,0.019173,0.043029,0.026330,0.976977,0.986987,0.010010,0.489801,1,200,250
35,0.3,0.636637,0.997998,0.361361,0.382062,0.192198,0.132881,0.325079,0.248786,0.023528,0.018331,0.041859,0.024081,0.979980,0.988989,0.009009,0.477317,1,220,250


In [27]:
exp_dir = "../../../../data/ptest_arr_toxicity/"
ci_path = f"{exp_dir}ci_2500"
ci_path = f"{exp_dir}ci_1000"
ci_path = f"{exp_dir}ci"

alt_ci_nk_dict = compress_pickle.load(f"{ci_path}/ci_level={confidence_level}_col={col}_500_dist={distortion}.pkl.lz4")
print(len(alt_ci_nk_dict['Accuracy']))
print(len(alt_ci_nk_dict['Accuracy'][0]))

5
35


In [10]:
# ci_dfs = []
# for dataset_info in datasets:
#     exp_dir = dataset_info['exp_dir']
    
#     # ci_path = f"{exp_dir}ci_2500"
#     ci_path = f"{exp_dir}ci_1000"
#     # ci_path = f"{exp_dir}ci"

#     alt_ci_nk_dict = compress_pickle.load(f"{ci_path}/ci_level={confidence_level}_col={col}_500_dist={distortion}.pkl.lz4")
#     # print(len(alt_ci_nk_dict['Accuracy']))

#     ci_metric_dfs_dict = {}

#     for metric in metrics_list:
#         ci_rows = []
#         for idx, (n_items, k_responses) in enumerate(params_list[nk_list_idx][:35]):
#             # print(idx, n_items, k_responses)
#             ci_lower, ci_upper = alt_ci_nk_dict[metric][nk_list_idx][idx][0]
#             mean_score = alt_ci_nk_dict[metric][nk_list_idx][idx][1]
#             ci_rows.append({'ci_lower':ci_lower, 'ci_upper':ci_upper, 'ci_width':ci_upper-ci_lower, 'mean_score':mean_score, 'N':n_items, 'K':k_responses})
#             # break

#         ci_metric_dfs_dict[metric] = pd.DataFrame(ci_rows)

#     ci_dfs.append(ci_metric_dfs_dict)

# print(len(ci_dfs))


In [11]:
base_path = "output/tables_min_nk"
# base_path = f"output/tables_{nk_list[nk_list_idx]}"
if not os.path.exists(base_path):
    os.makedirs(base_path)

# table_name = os.path.join(base_path, "table.tex")
# df.to_latex(table_name, index=False, float_format="%.4f")

In [12]:
results_list = []
for idx, df in enumerate(all_dfs):
    results_nk = {'Dataset': f"{datasets[idx]['dataset']} (M={datasets[idx]['num_categories']})", 'Stat':'NK'}
    results_k = {'Dataset': f"{datasets[idx]['dataset']} (M={datasets[idx]['num_categories']})", 'Stat':'K'}
    results_delta = {'Dataset': f"{datasets[idx]['dataset']} (M={datasets[idx]['num_categories']})", 'Stat':'$\\Delta$'}
    
    for metric in metrics_list:
        ls = df[f'{metric}\\_p-value']
        met_df = df.iloc[ls.index[ls<=0.05]]
        rows_nk = met_df['NK']
        
        if len(rows_nk)==0:
            results_nk[metric] = '-'
            results_k[metric] = '-'
            results_delta[metric] = '-'
        else:
            min_nk = rows_nk.min()
            nk_df = met_df[met_df['NK']==min_nk]
            min_k = nk_df['K'].min()

            results_nk[metric] = min_nk
            results_k[metric] = min_k
            results_delta[metric] = df.iloc[nk_df['K'].idxmin()][f'{metric}\\_$\\Delta$']
    results_list.append(results_nk)
    results_list.append(results_k)
    results_list.append(results_delta)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df

,Dataset,Stat,Accuracy,MAE,Wins,KL-Div
0,Toxicity (M=2),NK,2500.000000,1000.000000,2500.000000,1000.000000
1,Toxicity (M=2),K,1.000000,9.000000,1.000000,100.000000
2,Toxicity (M=2),$\Delta$,0.040474,0.040594,101.185185,0.039947
3,DICES (M=3),NK,1000.000000,500.000000,1000.000000,1000.000000
4,DICES (M=3),K,1.000000,10.000000,1.000000,1.000000
5,DICES (M=3),$\Delta$,0.054561,0.038486,54.560561,1.507564
6,D3code (M=2),NK,2500.000000,1000.000000,2500.000000,1000.000000
7,D3code (M=2),K,1.000000,20.000000,1.000000,60.000000
8,D3code (M=2),$\Delta$,0.032260,0.043898,80.650651,0.036784
9,JobsQ1 (M=5),NK,250.000000,250.000000,250.000000,250.000000


In [ ]:
table_name = os.path.join(base_path, f"low_k_for_p_lte_05_nk.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

In [14]:
results_list = []
for idx, df in enumerate(all_dfs):
    results_pval = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={datasets[idx]['num_categories']})", 'Stat':'p-value'}
    results_nk = {'idx':idx, 'Dataset': f"{datasets[idx]['dataset']} (M={datasets[idx]['num_categories']})", 'Stat':'NK'}
    results_k = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={datasets[idx]['num_categories']}", 'Stat':'K'}
    results_delta = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={datasets[idx]['num_categories']}", 'Stat':'$\\Delta$'}

    for metric in metrics_list:
        results_pval[metric] = df[f'{metric}\\_p-value'].min()
        min_p_index = df[f'{metric}\\_p-value'].idxmin()
        k_value = df.iloc[min_p_index]['K']
        nk_value = df.iloc[min_p_index]['NK']
        results_k[metric] = int(k_value)
        results_nk[metric] = int(nk_value)
        results_delta[metric] = df.iloc[min_p_index][f'{metric}\\_$\\Delta$']
    results_list.append(results_pval)
    results_list.append(results_nk)
    results_list.append(results_k)
    results_list.append(results_delta)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df = res_df.set_index('idx')
res_df

,Dataset,Stat,Accuracy,MAE,Wins,KL-Div
idx,,,,,,
0,Toxicity (M=2),p-value,0.000000,0.000000,0.000000,0.000000
0,Toxicity (M=2),NK,25000.000000,5000.000000,25000.000000,10000.000000
0,Toxicity (M=2,K,1.000000,40.000000,1.000000,240.000000
0,Toxicity (M=2,$\Delta$,0.040456,0.057933,1011.405405,0.042909
1,DICES (M=3),p-value,0.000000,0.000000,0.000000,0.000000
1,DICES (M=3),NK,10000.000000,2500.000000,5000.000000,10000.000000
1,DICES (M=3,K,1.000000,60.000000,20.000000,1.000000
1,DICES (M=3,$\Delta$,0.053525,0.058701,87.572573,1.478937
2,D3code (M=2),p-value,0.000000,0.000000,0.000000,0.000000


In [ ]:
table_name = os.path.join(base_path, f"k_delta_low_p_nk.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f", multirow=True)

In [ ]:
# for idx, dataset_info in enumerate(datasets):
#     dataset = dataset_info['dataset']
#     sub_df = all_dfs[idx][['N', 'K', 'Accuracy\\_p-value', 'Accuracy\\_$\\Delta$', 'Accuracy\\_M1 GT Alt', 'Accuracy\\_M2 GT Alt']]
#     sub_df = sub_df[sub_df['K']<=100]
#     table_name = os.path.join(base_path, f"{dataset}_accuracy_table_k_100.tex")
#     sub_df.to_latex(table_name, index=False, float_format="%.4f")

In [ ]:
results_list = []
for idx, df in enumerate(all_dfs):
    results_pval = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'p-value'}
    results_k = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']}", 'Stat':'K'}
    results_delta = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']}", 'Stat':'$\\Delta$'}

    for metric in metrics_list:
        results_pval[metric] = df[f'{metric}\\_p-value'].min()
        min_p_index = df[f'{metric}\\_p-value'].idxmin()
        k_value = df.iloc[min_p_index]['K']
        results_k[metric] = int(k_value)
        results_delta[metric] = df.iloc[min_p_index][f'{metric}\\_$\\Delta$']
    results_list.append(results_pval)
    results_list.append(results_k)
    results_list.append(results_delta)
    # print(results)
    break

res_df = pd.DataFrame(results_list)
res_df = res_df.set_index('idx')
res_df

In [ ]:
table_name = os.path.join(base_path, f"k_delta_low_p_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f", multirow=True)

### K for lowest p-value

In [ ]:
results_list = []
for idx, df in enumerate(all_dfs):
    results = {}
    results['Dataset'] = f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})"
    for metric in metrics_list:
        min_p_index = df[f'{metric}\\_p-value'].idxmin()
        k_value = df.iloc[min_p_index]['K']
        results[f'{metric}\\_K'] = int(k_value)
        results[f'{metric}\\_$\\Delta$'] = df.iloc[min_p_index][f'{metric}\\_$\\Delta$']
    results_list.append(results)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df

In [ ]:
table_name = os.path.join(base_path, f"k_delta_for_low_p_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

### Lowest p-value

In [ ]:
results_list = []
for idx, df in enumerate(all_dfs):
    results = {}
    results['Dataset'] = f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})"
    for metric in metrics_list:
        results[f'{metric}\\_p-value'] = df[f'{metric}\\_p-value'].min()
    results_list.append(results)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df

In [ ]:
table_name = os.path.join(base_path, f"low_p_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

### Lowest K for p<=0.05

In [ ]:
results_list = []
for idx, df in enumerate(all_dfs):
    results = {'Dataset': f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'K'}
    results_delta = {'Dataset': f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'$\\Delta$'}
    
    for metric in metrics_list:
        ls = df[f'{metric}\\_p-value']
        rows = df.iloc[ls.index[ls<=0.05]]['K']
        
        if len(rows)==0:
            results[metric] = '-'
            results_delta[metric] = '-'
        else:
            results[metric] = rows.min()
            results_delta[metric] = df.iloc[rows.idxmin()][f'{metric}\\_$\\Delta$']
    results_list.append(results)
    results_list.append(results_delta)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df

In [ ]:
table_name = os.path.join(base_path, f"low_k_for_p_lte_05_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

### K for lowest ci-width

In [ ]:
results_list = []
for idx, ci_dict in enumerate(ci_dfs):
    results = {}
    results['Dataset'] = f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})"
    for metric in metrics_list:
        df = ci_dict[metric]
        min_p_index = df['ci_width'].idxmin()
        k_value = df.iloc[min_p_index]['K']
        results[f'{metric}\\_K'] = int(k_value)
        # results[f'{metric}\\_$\\Delta$'] = df.iloc[min_p_index]['mean_score']
    results_list.append(results)
    # print(results)
    # break

res_df = pd.DataFrame(results_list)
res_df

In [ ]:
table_name = os.path.join(base_path, f"k_for_low_ci_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

### Lowest ci-width

In [ ]:
results_list = []
for idx, ci_dict in enumerate(ci_dfs):
    results = {}
    results['Dataset'] = f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})"
    for metric in metrics_list:
        df = ci_dict[metric]
        results[f'{metric}\\_ci'] = df['ci_width'].min()
    results_list.append(results)
    # print(results)
    # break

res_df = pd.DataFrame(results_list)
res_df

In [ ]:
table_name = os.path.join(base_path, f"low_ci_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

In [ ]:
results_list = []
for idx, ci_dict in enumerate(ci_dfs):
    results_ci = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'ci-width'}
    results_k = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'K'}
    results_delta = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'$\\Delta$'}
    p_df = all_dfs[idx]

    for metric in metrics_list:
        df = ci_dict[metric]
        results_ci[metric] = df['ci_width'].min()
        min_p_index = df['ci_width'].idxmin()
        k_value = df.iloc[min_p_index]['K']
        results_k[metric] = int(k_value)
        # print(k_value, p_df.iloc[min_p_index]['K'])

        results_delta[metric] = p_df.iloc[min_p_index][f'{metric}\\_$\\Delta$']
    results_list.append(results_ci)
    results_list.append(results_k)
    results_list.append(results_delta)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df = res_df.set_index('idx')
res_df

In [ ]:
table_name = os.path.join(base_path, f"k_low_ci_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f", multirow=True)